In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

In [7]:
df = pd.read_csv('train.csv')

In [8]:
X = df.drop(columns=['id', 'accident_risk'])
y = df['accident_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# numerical_features = ['curvature', 'speed_limit', 'num_reported_accidents']
# categorical_features = ['lighting', 'weather']
# boolean_features = ['public_road', 'holiday']

# numerical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy="mean")),
#     ('scaler', StandardScaler())
# ])

# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy="most_frequent")),
#     ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
# ])

# boolean_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy="most_frequent"))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numerical_transformer, numerical_features),
#         ("cat", categorical_transformer, categorical_features),
#         ("bool", boolean_transformer, boolean_features),
#     ],
#     remainder='drop'
# )

In [9]:
numerical_features = ['curvature', 'speed_limit', 'num_reported_accidents']
categorical_features = ['lighting', 'weather']
boolean_features = ['public_road', 'holiday']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

boolean_transformer = Pipeline(steps=[
    ('convert_bool', FunctionTransformer(lambda x: x.astype(float), validate=False)),
    ('imputer', SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
        ("bool", boolean_transformer, boolean_features),
    ],
    remainder='drop'
)

Enchanted_forest = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,   
        max_depth=10,       
        random_state=42     
    ))
])

Enchanted_forest.fit(X_train, y_train)

predictions = Enchanted_forest.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, predictions))

print(f"Random Forest RMSE: {rmse:.4f}")

Random Forest RMSE: 0.0556


In [10]:
# Enchanted_forest = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(
#         n_estimators=100,   
#         max_depth=10,       
#         random_state=42     
#     ))
# ])

# Enchanted_forest.fit(X_train, y_train)

# predictions = Enchanted_forest.predict(X_train)
# rmse = np.sqrt(mean_squared_error(y_train, predictions))

# print(f"Random Forest RMSE: {rmse:.4f}")

## **Enchanted forest RMSE 0.0556**

In [11]:
final_df = pd.read_csv('test.csv')

Enchanted_forest_predictions = Enchanted_forest.predict(final_df.drop(columns=['id']))

submission = pd.DataFrame({
    'id': final_df['id'],
    'accident_risk': Enchanted_forest_predictions
})

submission.to_csv('submission_enchanted_forest.csv', index=False)

## **Fine-tune the model**

In [12]:
# base_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(random_state=42))
# ])

# param_grid = {
#     'regressor__n_estimators': [100, 200, 300],
#     'regressor__max_depth': [None, 10, 20],
#     'regressor__min_samples_split': [2, 5, 10],
#     'regressor__max_features': ['sqrt', 'log2']
# }

# grid_search = GridSearchCV(
#     estimator=base_pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring='neg_root_mean_squared_error',
#     n_jobs=-1
# )

# grid_search.fit(X_train, y_train)

# Enchanted_forest_finetuned = grid_search.best_estimator_
# y_pred = Enchanted_forest_finetuned.predict(X_test)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# print(f"Best Parameters: {grid_search.best_params_}")
# print(f"Final RMSE: {rmse:.4f}")